# 05 — Enrichissement météo (OpenData) : apporte-t-il quelque chose ?

**Données** : normales climatiques 1991-2020 par commune (température moyenne annuelle, précipitations annuelles), préparées par `preparer_meteo_corse.py` et rattachées aux logements par `src/data/meteo.py`. Voir `SOURCE_METEO.md` pour la provenance .

**Question** : ces deux variables améliorent-elles la prédiction de l'étiquette (classification) et de la consommation (régression) ?

**Précaution** : le DPE est calculé avec un climat de référence défini par zone climatique et par altitude, pas avec la météo réelle de la commune. On s'attend donc à un gain faible. De plus, une valeur météo est **identique pour tous les logements d'une commune** : elle peut servir de simple *identifiant de commune* et capter des effets locaux (types de logements, pratiques de diagnostic) sans rapport avec le climat. Pour distinguer les deux, on ajoute un **témoin** : la même météo, mais **mélangée aléatoirement entre les communes**. Si la vraie météo ne fait pas mieux que le témoin, son apport n'est pas climatique.

**Démarche** : validation croisée groupée par adresse sur le train, avec les mêmes plis pour toutes les variantes ; le test n'est consulté que si la décision est favorable.

## 0. Préparation

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import (accuracy_score, f1_score, mean_absolute_error, r2_score)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))

from src.utilis import config
from src.data import enrichissement_meteo as meteo
from src.data import transformateurs as tf

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
plt.rcParams["figure.figsize"] = (9, 4.5)

DOSSIER_FIG = config.RACINE / "reports" / "figures"
ETIQ, CONSO = config.CIBLE_CLASSIFICATION, config.CIBLE_REGRESSION
ORDRE = config.ORDRE_ETIQUETTES
GRAINE = config.GRAINE_ALEATOIRE
N_PLIS = 5
N_TEMOINS = 3        # nombre de mélanges aléatoires de la météo entre communes


def sauver(fig, nom):
    DOSSIER_FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(DOSSIER_FIG / f"{nom}.png", dpi=150, bbox_inches="tight")

## 1. Chargement et rattachement de la météo

In [2]:
train = pd.read_csv(config.DATA_TRAITE / "train.csv", low_memory=False)
test = pd.read_csv(config.DATA_TRAITE / "test.csv", low_memory=False)
with open(config.DOSSIER_MODELES / "variables_retenues.json", encoding="utf-8") as f:
    decisions = json.load(f)

VAR_NUM = decisions["variables_numeriques"]
VAR_CAT = decisions["variables_categorielles"]
VARIABLES = VAR_NUM + VAR_CAT

table_meteo = meteo.charger_table_meteo()
train = meteo.ajouter_meteo(train, table_meteo)     # lève une erreur si une commune n'a pas de météo
test = meteo.ajouter_meteo(test, table_meteo)
for d in (train, test):
    d["groupe"] = d["adresse_ban"].fillna(d["numero_dpe"])
    d["conso_aberrante"] = d["conso_aberrante"].astype(str).str.lower().eq("true")

assert not set(train["groupe"]) & set(test["groupe"]), "des adresses sont dans le train ET le test"
assert len(train) == train["numero_dpe"].nunique()

print(f"Table météo : {len(table_meteo)} communes")
print(f"train : {len(train):,} lignes | test : {len(test):,} lignes | toutes rattachées à une commune de la table")
print(f"{train['nom_commune_ban'].nunique()} communes différentes dans le train")

Table météo : 360 communes
train : 39,387 lignes | test : 9,839 lignes | toutes rattachées à une commune de la table
337 communes différentes dans le train


## 2. Que contiennent ces variables ?
On regarde leur variété, leur lien avec l'altitude et le zonage climatique du DPE (déjà dans le modèle), et leur lien brut avec les cibles.

In [3]:
meteo_vars = meteo.COLONNES_METEO
print(train[meteo_vars].describe(percentiles=[0.05, 0.5, 0.95]).round(1).to_string())

par_commune = train.groupby("nom_commune_ban").size().sort_values(ascending=False)
print(f"\nLes 10 communes les plus représentées pèsent {par_commune.head(10).sum() / len(train):.0%} des logements du train.")
print(f"Valeurs de température différentes : {train['temperature_moyenne_c'].nunique()} (une par commune)")

       temperature_moyenne_c  precipitation_annuelle_mm
count             39,387.000                 39,387.000
mean                  15.800                    809.100
std                    1.100                     93.800
min                   11.000                    595.000
5%                    13.600                    660.100
50%                   16.200                    828.800
95%                   17.100                    929.800
max                   17.200                  1,016.700

Les 10 communes les plus représentées pèsent 56% des logements du train.
Valeurs de température différentes : 251 (une par commune)


In [4]:
for c in ["classe_altitude", "zone_climatique"]:
    if c in train.columns:
        print(f"--- température moyenne selon {c} ---")
        print(train.groupby(c)["temperature_moyenne_c"].agg(["count", "mean", "min", "max"]).round(1).to_string(), "\n")

ref = train.loc[~train["conso_aberrante"]]
print("Corrélation de Spearman avec la consommation :")
for c in meteo_vars:
    print(f"  {c} : {ref[c].corr(ref[CONSO], method='spearman'):.3f}")
print("Corrélation entre les deux variables météo :",
      round(train[meteo_vars[0]].corr(train[meteo_vars[1]], method="spearman"), 3))

--- température moyenne selon classe_altitude ---
                  count   mean    min    max
classe_altitude                             
400-800m           2607 14.600 11.500 17.100
inférieur à 400m  35960 15.900 11.900 17.200
supérieur à 800m    189 13.900 11.000 17.100 

--- température moyenne selon zone_climatique ---
                 count   mean    min    max
zone_climatique                            
H1a                209 15.400 12.400 17.200
H1b                 32 16.000 12.700 17.100
H1c                 46 16.200 12.700 17.200
H2a                  8 16.000 14.300 16.500
H2b                 34 14.900 13.900 17.200
H2c                 37 16.000 14.300 17.200
H2d                 32 15.700 13.900 17.200
H3               38358 15.800 11.000 17.200 

Corrélation de Spearman avec la consommation :
  temperature_moyenne_c : -0.113
  precipitation_annuelle_mm : 0.057
Corrélation entre les deux variables météo : -0.323


## 3. Le témoin : la même météo, mélangée entre les communes
Chaque commune reçoit la météo d'une autre commune tirée au hasard. La distribution des valeurs reste identique, mais le lien avec le vrai climat local est détruit. Tout gain obtenu par ce témoin vient de l'identification de la commune, pas du climat.

In [5]:
cle_train = meteo.creer_cle(train["code_departement_ban"], train["nom_commune_ban"])
cle_test = meteo.creer_cle(test["code_departement_ban"], test["nom_commune_ban"])
communes = np.array(sorted(set(cle_train) | set(cle_test)))
valeurs = table_meteo.set_index("cle").loc[communes, meteo_vars]

COLONNES_TEMOINS = []
for k in range(N_TEMOINS):
    permutation = np.random.default_rng(GRAINE + k).permutation(len(communes))
    melange = pd.DataFrame(valeurs.to_numpy()[permutation], index=communes, columns=meteo_vars)
    for colonne in meteo_vars:
        nom = f"{colonne}_temoin{k + 1}"
        train[nom] = cle_train.map(melange[colonne]).to_numpy()
        test[nom] = cle_test.map(melange[colonne]).to_numpy()
    COLONNES_TEMOINS.append([f"{c}_temoin{k + 1}" for c in meteo_vars])

print(f"{N_TEMOINS} témoins créés :", COLONNES_TEMOINS)
print("Corrélation vraie / mélangée (doit être proche de 0) :",
      round(train[meteo_vars[0]].corr(train[COLONNES_TEMOINS[0][0]]), 3))

3 témoins créés : [['temperature_moyenne_c_temoin1', 'precipitation_annuelle_mm_temoin1'], ['temperature_moyenne_c_temoin2', 'precipitation_annuelle_mm_temoin2'], ['temperature_moyenne_c_temoin3', 'precipitation_annuelle_mm_temoin3']]
Corrélation vraie / mélangée (doit être proche de 0) : -0.271


## 4. Variantes évaluées par validation croisée (train uniquement)
Modèle commun aux deux tâches : le **gradient boosting** (réglages par défaut). Ce n'est pas nécessairement le modèle final : l'objectif est de comparer proprement avec et sans météo, à modèle et plis identiques.

| Variante | Variables ajoutées |
|---|---|
| référence | aucune |
| température | `temperature_moyenne_c` |
| précipitations | `precipitation_annuelle_mm` |
| les deux | les deux variables |
| témoins 1 à 3 | les deux variables, mélangées entre communes |

In [6]:
VARIANTES = {
    "référence": [],
    "température": ["temperature_moyenne_c"],
    "précipitations": ["precipitation_annuelle_mm"],
    "les deux": list(meteo_vars),
}
for k, colonnes in enumerate(COLONNES_TEMOINS):
    VARIANTES[f"témoin {k + 1} (météo mélangée)"] = colonnes


def modele(tache, variables_num, variables_cat):
    prep = tf.construire_preprocesseur(variables_num, variables_cat, pour="arbres")
    if tache == "classification":
        m = HistGradientBoostingClassifier(class_weight="balanced", early_stopping=False, random_state=GRAINE)
    else:
        m = HistGradientBoostingRegressor(early_stopping=False, random_state=GRAINE)
    return Pipeline([("prep", prep), ("modele", m)])


def valider(tache, donnees, plis_t, extras):
    num, cat = VAR_NUM + extras, VAR_CAT
    colonnes = num + cat
    cible = ETIQ if tache == "classification" else CONSO
    scores = []
    for i_fit, i_val in plis_t:
        fit_v, val_v = donnees.iloc[i_fit], donnees.iloc[i_val]
        m = modele(tache, num, cat).fit(fit_v[colonnes], fit_v[cible])
        pred_v = m.predict(val_v[colonnes])
        if tache == "classification":
            scores.append(f1_score(val_v[cible], pred_v, average="macro", labels=ORDRE, zero_division=0))
        else:
            scores.append(mean_absolute_error(val_v[cible], pred_v))
    return np.array(scores)


cv = StratifiedGroupKFold(n_splits=N_PLIS, shuffle=True, random_state=GRAINE)
train_cls = train.reset_index(drop=True)
plis_cls = list(cv.split(train_cls, train_cls[ETIQ], groups=train_cls["groupe"]))

train_reg = train.loc[~train["conso_aberrante"]].reset_index(drop=True)
plis_reg = list(cv.split(train_reg, train_reg[ETIQ], groups=train_reg["groupe"]))
print("Plis de classification :", [len(v) for _, v in plis_cls])
print("Plis de régression     :", [len(v) for _, v in plis_reg])

Plis de classification : [7878, 7878, 7877, 7877, 7877]
Plis de régression     : [7877, 7877, 7877, 7877, 7877]


In [7]:
resultats = {"classification": {}, "regression": {}}
for nom, extras in VARIANTES.items():
    resultats["classification"][nom] = valider("classification", train_cls, plis_cls, extras)
    resultats["regression"][nom] = valider("regression", train_reg, plis_reg, extras)
    print(f"{nom} : terminé")

référence : terminé
température : terminé
précipitations : terminé
les deux : terminé
témoin 1 (météo mélangée) : terminé
témoin 2 (météo mélangée) : terminé
témoin 3 (météo mélangée) : terminé


In [8]:
def tableau(tache, meilleur_si_grand):
    lignes = {}
    ref_scores = resultats[tache]["référence"]
    for nom, scores in resultats[tache].items():
        lignes[nom] = {"moyenne": scores.mean(), "ecart_type_entre_plis": scores.std()}
    t = pd.DataFrame(lignes).T
    ecart = t["moyenne"] - t.loc["référence", "moyenne"]
    t["gain_par_rapport_a_la_reference"] = ecart if meilleur_si_grand else -ecart   # positif = mieux
    t["gain_en_ecarts_types"] = t["gain_par_rapport_a_la_reference"] / t.loc["référence", "ecart_type_entre_plis"]
    return t


tab_cls = tableau("classification", meilleur_si_grand=True)
tab_reg = tableau("regression", meilleur_si_grand=False)
print("Classification : F1 macro en validation croisée (gain positif = mieux)")
display(tab_cls)
print("Régression : MAE en validation croisée (gain positif = erreur plus faible)")
tab_reg

Classification : F1 macro en validation croisée (gain positif = mieux)


,moyenne,ecart_type_entre_plis,gain_par_rapport_a_la_reference,gain_en_ecarts_types
référence,0.519,0.007,0.000,0.000
température,0.528,0.010,0.009,1.214
précipitations,0.529,0.008,0.010,1.309
les deux,0.528,0.008,0.009,1.211
témoin 1 (météo mélangée),0.522,0.008,0.003,0.435
témoin 2 (météo mélangée),0.519,0.009,0.000,0.051
témoin 3 (météo mélangée),0.526,0.007,0.008,1.012


Régression : MAE en validation croisée (gain positif = erreur plus faible)


,moyenne,ecart_type_entre_plis,gain_par_rapport_a_la_reference,gain_en_ecarts_types
référence,27.550,0.836,-0.000,-0.000
température,27.414,1.015,0.136,0.163
précipitations,27.339,0.702,0.211,0.253
les deux,27.318,0.910,0.232,0.278
témoin 1 (météo mélangée),27.486,0.936,0.064,0.076
témoin 2 (météo mélangée),27.346,0.835,0.204,0.244
témoin 3 (météo mélangée),27.305,0.802,0.245,0.293


**Lecture** : le gain est exprimé en écarts-types d'un pli à l'autre. Un gain inférieur à 1 écart-type n'est pas net. Le vrai test est la comparaison avec les **témoins** : si la ligne « les deux » ne dépasse pas nettement les témoins, la météo ne sert qu'à identifier la commune.

## 5. Décision

In [9]:
def decider(tab):
    gain_vrai = tab.loc["les deux", "gain_par_rapport_a_la_reference"]
    gains_temoins = tab.loc[[n for n in tab.index if n.startswith("témoin")], "gain_par_rapport_a_la_reference"]
    seuil_bruit = tab.loc["référence", "ecart_type_entre_plis"]
    adopter = bool(gain_vrai > seuil_bruit and gain_vrai > gains_temoins.max())
    return adopter, gain_vrai, gains_temoins.max(), seuil_bruit


decision = {}
for tache, tab in [("classification", tab_cls), ("regression", tab_reg)]:
    adopter, gain, gain_temoin, bruit = decider(tab)
    decision[tache] = {"adopter": adopter, "gain": float(gain), "meilleur_temoin": float(gain_temoin),
                       "ecart_type_entre_plis": float(bruit)}
    print(f"{tache} : gain de la vraie météo = {gain:.4f} | meilleur témoin = {gain_temoin:.4f} | "
          f"bruit (écart-type entre plis) = {bruit:.4f} -> {'ADOPTER' if adopter else 'NE PAS adopter'}")
print("\nRègle : adopter si le gain dépasse à la fois le bruit entre plis et le meilleur des témoins.")

classification : gain de la vraie météo = 0.0090 | meilleur témoin = 0.0075 | bruit (écart-type entre plis) = 0.0074 -> ADOPTER
regression : gain de la vraie météo = 0.2320 | meilleur témoin = 0.2448 | bruit (écart-type entre plis) = 0.8358 -> NE PAS adopter

Règle : adopter si le gain dépasse à la fois le bruit entre plis et le meilleur des témoins.


## 6. Évaluation sur le test (uniquement si la météo est adoptée)
Une seule fois, pour les tâches où la décision est favorable : même modèle avec et sans météo, ajustés sur tout le train.

In [10]:
resume_test = {}
if not any(d["adopter"] for d in decision.values()):
    print("Décision défavorable pour les deux tâches : le test n'est pas consulté.")
else:
    for tache, d in decision.items():
        if not d["adopter"]:
            continue
        cible = ETIQ if tache == "classification" else CONSO
        donnees_fit = train if tache == "classification" else train.loc[~train["conso_aberrante"]]
        donnees_test = test if tache == "classification" else test.loc[~test["conso_aberrante"]]
        lignes_t = {}
        for nom, extras in {"sans météo": [], "avec météo": list(meteo_vars)}.items():
            num, cat = VAR_NUM + extras, VAR_CAT
            m = modele(tache, num, cat).fit(donnees_fit[num + cat], donnees_fit[cible])
            p = m.predict(donnees_test[num + cat])
            if tache == "classification":
                lignes_t[nom] = {"f1_macro": f1_score(donnees_test[cible], p, average="macro", labels=ORDRE, zero_division=0),
                                 "exactitude": accuracy_score(donnees_test[cible], p)}
            else:
                lignes_t[nom] = {"mae": mean_absolute_error(donnees_test[cible], p), "r2": r2_score(donnees_test[cible], p)}
        resume_test[tache] = pd.DataFrame(lignes_t).T
        print(f"--- {tache}, test ---")
        display(resume_test[tache])

--- classification, test ---


,f1_macro,exactitude
sans météo,0.513,0.628
avec météo,0.521,0.630


## 7. Sauvegarde

In [11]:
config.DATA_TRAITE.mkdir(parents=True, exist_ok=True)
tab_cls.to_csv(config.DATA_TRAITE / "meteo_variantes_classification.csv")
tab_reg.to_csv(config.DATA_TRAITE / "meteo_variantes_regression.csv")

bilan_meteo = {
    "source": "normales climatiques 1991-2020 par commune (CC BY 4.0), voir SOURCE_METEO.md",
    "variables": meteo_vars,
    "decision": decision,
    "test": {t: df_t.to_dict("index") for t, df_t in resume_test.items()},
}
with open(config.DOSSIER_MODELES / "decision_meteo.json", "w", encoding="utf-8") as f:
    json.dump(bilan_meteo, f, ensure_ascii=False, indent=2)
print("Fichiers écrits : meteo_variantes_classification.csv, meteo_variantes_regression.csv, decision_meteo.json")

Fichiers écrits : meteo_variantes_classification.csv, meteo_variantes_regression.csv, decision_meteo.json
